<div style='background:#1a3a5c;color:white;padding:22px 30px;border-radius:10px;font-family:sans-serif'>
<h1 style='margin:0 0 6px 0;font-size:1.6em'>⚡ DESAFÍO RELÁMPAGO — Sesión 06 · SOLUCIONES (Profesor)</h1>
<h2 style='margin:0 0 10px 0;font-weight:300;font-size:1.1em'>Phase Vocoder: Congelar el Tiempo</h2>
<p style='margin:0;opacity:0.8;font-size:0.95em'>
Incluye: código completo, respuestas esperadas a predicciones,
notas de facilitación y diagnóstico de errores frecuentes.
</p></div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft as scipy_stft, istft as scipy_istft
import wave, os

FS = 44100

def save_wav(filename, audio, fs=FS):
    audio = np.array(audio, dtype=np.float64)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak * 0.88
    pcm = np.clip(audio * 32767, -32768, 32767).astype(np.int16)
    with wave.open(filename, 'w') as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(fs)
        wf.writeframes(pcm.tobytes())
    print(f'  Guardado: {filename}  ({os.path.getsize(filename)/1024:.0f} KB)')

# Vocal sintética
dur = 2.0
t = np.linspace(0, dur, int(dur * FS), endpoint=False)
f0 = 150
vocal = sum((1.0/k) * np.sin(2*np.pi*f0*k*t) for k in range(1, 7))
vocal = vocal / np.max(np.abs(vocal)) * 0.8
save_wav('dr06_vocal_original.wav', vocal)

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(t[:2000], vocal[:2000], color='steelblue', lw=0.8)
axes[0].set(xlabel='Tiempo (s)', title='Vocal sintética — 150 Hz + armónicos')
axes[1].specgram(vocal, NFFT=2048, Fs=FS, noverlap=1536, cmap='magma')
axes[1].set_ylim(0, 2000)
axes[1].set(xlabel='Tiempo (s)', ylabel='Hz', title='Espectrograma')
plt.tight_layout(); plt.show()

## 1 · Predicciones — Respuestas esperadas

**Pregunta 1:** Si tomamos la fase cruda `φ(m+1, k)` y la usamos directamente (sin propagar), ¿sigue siendo 150 Hz?  
→ **No necesariamente.** La fase de un bin en dos frames consecutivos es la fase acumulada de la señal, que tiene una relación no-lineal con la frecuencia. Lo que sí define la frecuencia es la *diferencia de fase entre frames* dividida por el tiempo entre ellos.

**Pregunta 2:** Acumular la fase cruda `+=` es sumar ángulos absolutos — como si dijera "la posición acumulada del oscilador es la suma de todas sus posiciones anteriores". Eso es físicamente incorrecto. Lo correcto es acumular la *velocidad angular* (frecuencia instantánea).

**Pregunta 3:** Efecto esperado: "phasiness" o flanging metálico — las armónicas de una nota musical dejan de estar en fase entre sí, produciéndose interferencias constructivas/destructivas que ruedan a través de la escala de frecuencias.

**Nota de facilitación:** El error más común en la predicción es decir "el pitch baja" (confundiendo con el método naive de bajar el samplerate). El Phase Vocoder buggy no baja el pitch — produce artefactos de fase, que es algo distinto. Escuchar el archivo `dr06_naive_stretch.wav` ayuda a anclar esta distinción.

In [ ]:
N_FFT, RA, ALPHA = 2048, 512, 4.0
RS = int(RA * ALPHA)

_, _, X = scipy_stft(vocal, fs=FS, nperseg=N_FFT, noverlap=N_FFT-RA,
                     window='hann', boundary='zeros')

# Síntesis ingenua: reconstruir con Rs sin corregir la fase
_, vocal_naive = scipy_istft(X, fs=FS, nperseg=N_FFT, noverlap=N_FFT-RS,
                              window='hann', boundary='zeros')
save_wav('dr06_naive_stretch.wav', vocal_naive)

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].specgram(vocal, NFFT=2048, Fs=FS, noverlap=1536, cmap='magma')
axes[0].set_ylim(0, 2000); axes[0].set(xlabel='s', ylabel='Hz', title='Original')
axes[1].specgram(vocal_naive, NFFT=2048, Fs=FS, noverlap=1536, cmap='magma')
axes[1].set_ylim(0, 2000); axes[1].set(xlabel='s', ylabel='Hz', title='Naive stretch — nótese el desorden de fase')
plt.tight_layout(); plt.show()

## 2 · Phase Unwrapping — Solución Completa

**Clave pedagógica:** El Paso 3 (`princ_arg`) es el que más estudiantes omiten. Sin él, la diferencia de fase residual puede ser mayor que π, lo que causa saltos de fase espurios en la acumulación. `princArg` "envuelve" el ángulo al rango canónico `[-π, π]`, que corresponde al rango de frecuencias `[-fs/2, fs/2]`.

**Por qué `(omega + delta_phi_residual) * rs / ra` preserva el pitch:**  
- `omega + delta_phi_residual` = la frecuencia instantánea real del oscilador en el bin `k`  
- Multiplicar por `rs/ra = alpha` escala el *tiempo* de avance, pero la *frecuencia* sigue siendo la misma  
- Resultado: el oscilador avanza más lento en el tiempo, pero a la misma frecuencia → mismo pitch, menor velocidad

In [ ]:
def princ_arg(x):
    """[WHY] Envuelve ángulo a [-π, π] — sin esto la fase acumulada diverge."""
    return ((x + np.pi) % (2 * np.pi)) - np.pi


def phase_vocoder(signal, alpha, n_fft=2048, ra=512):
    """
    Phase Vocoder con Phase Unwrapping completo.
    alpha > 1 → más lento  |  alpha < 1 → más rápido
    """
    rs = int(ra * alpha)  # [WHY] hop de síntesis mayor → signal más larga → más lenta

    _, _, X = scipy_stft(signal, fs=FS, nperseg=n_fft, noverlap=n_fft - ra,
                         window='hann', boundary='zeros')
    n_bins, n_frames = X.shape

    # [WHY] Ω_k = fase que avanzaría el bin k exactamente en Ra muestras si su
    # frecuencia fuera k·fs/N. Es el avance "esperado". La desviación respecto
    # a esto es la frecuencia instantánea real.
    omega = 2 * np.pi * np.arange(n_bins) * ra / n_fft

    phase_out = np.zeros_like(X)
    phase_synth = np.angle(X[:, 0])  # inicializar con la fase del primer frame
    phase_out[:, 0] = X[:, 0]
    phase_prev = np.angle(X[:, 0])

    for m in range(1, n_frames):
        mag = np.abs(X[:, m])
        phase_curr = np.angle(X[:, m])

        # Paso 1: diferencia de fase cruda
        delta_phi = phase_curr - phase_prev

        # Paso 2+3: restar avance esperado y envolver — esto da la frecuencia instantánea real
        # [WHY] sin princ_arg, delta_phi puede ser > π → la frecuencia parecería de otro bin
        delta_phi_residual = princ_arg(delta_phi - omega)

        # Paso 4: acumular fase de síntesis escalada
        # [WHY] (omega + residual) es la frecuencia instantánea; * rs/ra escala el tiempo
        phase_synth = phase_synth + (omega + delta_phi_residual) * rs / ra

        phase_out[:, m] = mag * np.exp(1j * phase_synth)
        phase_prev = phase_curr

    _, output = scipy_istft(phase_out, fs=FS, nperseg=n_fft, noverlap=n_fft - rs,
                            window='hann', boundary='zeros')
    return output


# Verificación: alpha=1.0 debe ser casi idéntico al original
out_1x = phase_vocoder(vocal, alpha=1.0)
l = min(len(vocal), len(out_1x))
err = np.max(np.abs(vocal[:l] - out_1x[:l]))
print(f'Error alpha=1.0: {err:.4f}  (< 0.05 es bueno para Hann con overlap)' )
save_wav('dr06_pv_alpha1.wav', out_1x)
print('✓ Phase Vocoder OK')

## 3 · El Bug — Análisis

```python
phase_acc += phase_raw  # ← BUG
```

**El bug está en los Pasos 1–3 simultáneamente:**
- Está acumulando la **fase absoluta** del frame actual (`phase_raw = angle(X[:,m])`), no la *diferencia de fase*.
- No resta el avance esperado `Ω_k`, así que no aísla la desviación residual.
- No aplica `princArg`, así que la acumulación diverge gradualmente.

El resultado es que cada bin acumula una fase aleatoria dependiente del contenido de la señal, no de su frecuencia real. Las armónicas relacionadas entre sí dejan de estar en fase → **flanging metálico** ("phasiness").

**Nota de facilitación:** Preguntar al grupo: ¿en qué línea exactamente está el bug? La respuesta esperada es `phase_acc += phase_raw`. Luego: ¿cuál sería la línea correcta? `phase_acc += (omega + princ_arg(phase_raw - phase_prev - omega)) * rs/ra`.

El grupo suele confundirse entre "bug en el cálculo" y "bug en el acumulador". Aclarar que el problema es no separar el avance *esperado* del avance *real*.

In [ ]:
def phase_vocoder_buggy(signal, alpha=4.0, n_fft=2048, ra=512):
    """
    Versión con el bug clásico — acumula fase cruda sin unwrapping.
    Genera el 'slowmo_roto.wav' de la Fun Task.
    """
    rs = int(ra * alpha)
    _, _, X = scipy_stft(signal, fs=FS, nperseg=n_fft, noverlap=n_fft - ra,
                         window='hann', boundary='zeros')
    phase_acc = np.angle(X[:, 0])
    output = [X[:, 0]]
    for m in range(1, X.shape[1]):
        mag = np.abs(X[:, m])
        phase_raw = np.angle(X[:, m])
        phase_acc += phase_raw   # ← el bug: suma fases absolutas, no velocidades angulares
        output.append(mag * np.exp(1j * phase_acc))
    _, y = scipy_istft(np.array(output).T, fs=FS, nperseg=n_fft, noverlap=n_fft - rs,
                       window='hann', boundary='zeros')
    return y


out_correct = phase_vocoder(vocal, alpha=4.0)
out_buggy   = phase_vocoder_buggy(vocal, alpha=4.0)

save_wav('dr06_correcto_alpha4.wav', out_correct)
save_wav('dr06_buggy_alpha4.wav',   out_buggy)

fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, (title, sig) in zip(axes, [
    ('Original (1x)', vocal),
    ('Correcto (4x, Phase Unwrapping)', out_correct),
    ('Buggy (4x, sin Phase Unwrapping)', out_buggy),
]):
    ax.specgram(sig[:min(len(sig), FS*10)], NFFT=2048, Fs=FS, noverlap=1536, cmap='magma')
    ax.set_ylim(0, 2000)
    ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('Hz'); ax.set_title(title)
plt.suptitle('Correcto: líneas horizontales rectas · Buggy: líneas con desorden vertical', y=1.03)
plt.tight_layout(); plt.show()

print('DIAGNÓSTICO VISUAL: en la versión buggy, las líneas horizontales de')
print('los armónicos aparecen "flameadas" o con fluctuaciones verticales.')
print('Eso es la incoherencia de fase entre parciales relacionados.')

In [ ]:
# Exploración de alpha — varios factores
print('Generando variantes de tiempo...')
for alpha, label in [(0.5, '050'), (2.0, '200'), (4.0, '400'), (10.0, '1000')]:
    # n_fft=4096 con alpha grande produce texturas más etéreas (ventana larga)
    n_fft_val = 4096 if alpha >= 4.0 else 2048
    ra_val = n_fft_val // 4
    out = phase_vocoder(vocal, alpha=alpha, n_fft=n_fft_val, ra=ra_val)
    save_wav(f'dr06_alpha{label}.wav', out)
    print(f'  alpha={alpha:4.1f} → duración: {len(out)/FS:.2f}s  (n_fft={n_fft_val})')

print()
print('OBSERVACIÓN: con alpha=10 y n_fft=4096 se obtiene el efecto Paulstretch.')
print('Los estudiantes suelen encontrar el umbral "voz→textura" alrededor de alpha=6-8.')

## Notas de Facilitación

### Errores frecuentes en la implementación

1. **Olvidar `princ_arg` en el Paso 3.** Síntoma: la versión con alpha=1 ya suena metálica. El estudiante puede pensar que el algoritmo funciona, pero si alpha=1 no reproduce la señal original, hay un problema en la acumulación.

2. **Usar `rs` en lugar de `rs/ra` en el Paso 4.** Síntoma: el pitch sí cambia al estirar. La multiplicación correcta es `(omega + delta_phi_residual) * rs / ra` (o equivalentemente `* alpha`), no solo `* rs`.

3. **Inicializar `phase_synth` con ceros en lugar de `angle(X[:,0])`.** Síntoma: el primer frame del output suena como ruido o silencio. El acumulador debe arrancar desde la fase real del primer frame.

4. **Confundir el hop de síntesis en `scipy_istft`.** El `noverlap` de la istft debe calcularse como `n_fft - rs`, no `n_fft - ra`. Si se usa `ra` en la síntesis, no hay estiramiento.

5. **Bug del WAV float32** (clásico): usar `setsampwidth(4)` con float32 → ruido en macOS. Ya corregido en `save_wav()` con int16.

### Pregunta de cierre — Respuesta esperada

"¿A partir de qué α deja de sonar como voz?"

No hay respuesta única — depende del contenido del audio y del tamaño de ventana. Con `n_fft=2048`, la mayoría de los estudiantes identifica el umbral entre α=4 y α=8. Con `n_fft=4096`, el umbral sube porque la ventana larga promedia más tiempo en cada frame, produciendo una textura más fluida. Esta variabilidad es intencional — hace que el estudiante entienda que α y n_fft tienen efectos perceptivos independientes.

### Conexión con la Fun Task

La función `phase_vocoder()` de este desafío es literalmente el entregable principal de la Fun Task 06. Los estudiantes solo necesitan:
1. Reemplazar la vocal sintética por su audio de voz real
2. Usar `alpha=4.0` (el valor de la fun task)
3. Generar también la versión buggy para la "Autopsia Auditiva"
4. Añadir los comentarios `[WHY]` obligatorios